# 06 - Constrained IK With IPOPT And `pinocchio.casadi`

This notebook solves the UR5 position IK problem as one nonlinear program.

The end-effector position expression is built symbolically with `pinocchio.casadi`. Joint limits are enforced as hard variable bounds.

In [ ]:
!pip install -q condacolab

import condacolab
condacolab.install()


In [ ]:
!conda install -y -n base -c conda-forge python=3.12 pinocchio robot_descriptions casadi numpy matplotlib

In [ ]:
import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin
import pinocchio.casadi as cpin

from robot_descriptions.loaders.pinocchio import load_robot_description

ipopt_opts = {
    "ipopt.print_level": 0,
    "ipopt.sb": "yes",
    "ipopt.max_iter": 200,
    "print_time": False,
}


## Load UR5 And Define The Same Kind Of Target

The target is generated from a feasible UR5 configuration. The NLP tracks only the end-effector position.

In [ ]:
robot = load_robot_description("ur5_description")
model = robot.model
data = model.createData()

tool_frame_id = model.getFrameId("tool0")
assert tool_frame_id < len(model.frames), "Could not find frame tool0"

q_min_raw = model.lowerPositionLimit.copy()
q_max_raw = model.upperPositionLimit.copy()
q_min = np.where(np.isfinite(q_min_raw), q_min_raw, -2.0 * np.pi)
q_max = np.where(np.isfinite(q_max_raw), q_max_raw, 2.0 * np.pi)

q0 = pin.neutral(model)
q0 = np.clip(q0, q_min + 1e-3, q_max - 1e-3)
q_nominal = q0.copy()

q_target = q0 + np.array([0.4, -0.7, 0.5, -0.4, 0.3, 0.2])
q_target = np.clip(q_target, q_min + 0.05, q_max - 0.05)

pin.framesForwardKinematics(model, data, q_target)
desired_position = data.oMf[tool_frame_id].translation.copy()

pin.framesForwardKinematics(model, data, q0)
start_position = data.oMf[tool_frame_id].translation.copy()

obstacle_center = start_position + 0.5 * (desired_position - start_position)
obstacle_center[2] += 0.10
obstacle_radius = 0.08

print("start position:", start_position)
print("desired position:", desired_position)
print("obstacle center:", obstacle_center)
print("obstacle radius:", obstacle_radius)

## Nonlinear IK Formulation

The NLP uses the true nonlinear forward kinematics expression from `pinocchio.casadi`.

Decision variable:

$$
q \in \mathbb{R}^{n_q}.
$$

Symbolic end-effector position:

$$
p_{\mathrm{ee}}(q) \in \mathbb{R}^3.
$$

Target position from the desired pose:

$$
p_{\mathrm{des}} = \mathrm{translation}(T_{\mathrm{des}}).
$$

A spherical keep-out region is defined by center `c_obs` and radius `r_obs`. The true nonlinear inequality is

$$
\|p_{\mathrm{ee}}(q) - c_{\mathrm{obs}}\|_2^2 \ge r_{\mathrm{obs}}^2.
$$

The full NLP is

$$
\begin{aligned}
\min_{q \in \mathbb{R}^{n_q}} \quad
& \frac{1}{2}\|p_{\mathrm{ee}}(q) - p_{\mathrm{des}}\|_2^2
+ \frac{\alpha}{2}\|q - q_{\mathrm{nom}}\|_2^2 \\
\text{s.t.} \quad
& q_{\min} \le q \le q_{\max}, \\
& \|p_{\mathrm{ee}}(q) - c_{\mathrm{obs}}\|_2^2 \ge r_{\mathrm{obs}}^2.
\end{aligned}
$$

## Build Symbolic Forward Kinematics With `pinocchio.casadi`

The key point is that `ee_position` is an `SX` expression, not a numeric finite-difference approximation.

In [ ]:
cmodel = cpin.Model(model)
cdata = cmodel.createData()

q_sym = ca.SX.sym("q", model.nq)
target_position_sym = ca.SX.sym("target_position", 3)
q_nominal_sym = ca.SX.sym("q_nominal", model.nq)
obstacle_center_sym = ca.SX.sym("obstacle_center", 3)
obstacle_radius_sym = ca.SX.sym("obstacle_radius")
p_sym = ca.vertcat(target_position_sym, q_nominal_sym, obstacle_center_sym, obstacle_radius_sym)

cpin.framesForwardKinematics(cmodel, cdata, q_sym)
ee_position = cdata.oMf[tool_frame_id].translation

posture_weight = 1e-3
position_error = ee_position - target_position_sym
posture_error = q_sym - q_nominal_sym
objective = 0.5 * ca.sumsqr(position_error) + 0.5 * posture_weight * ca.sumsqr(posture_error)

obstacle_distance_squared = ca.sumsqr(ee_position - obstacle_center_sym)
obstacle_clearance_squared = obstacle_distance_squared - obstacle_radius_sym**2
g = ca.vertcat(obstacle_clearance_squared)

nlp = {"x": q_sym, "p": p_sym, "f": objective, "g": g}
ik_nlp_solver = ca.nlpsol("ik_nlp_solver", "ipopt", nlp, ipopt_opts)

ee_position_fun = ca.Function("ee_position_fun", [q_sym], [ee_position])
print("symbolic end-effector expression shape:", ee_position.shape)
print("solver input names:", ik_nlp_solver.name_in())

## Prepare Solver Parameters

The target position, nominal posture, obstacle center, and obstacle radius are passed through the NLP parameter vector `p`.


In [ ]:
p_num = ca.vertcat(
    ca.DM(desired_position),
    ca.DM(q_nominal),
    ca.DM(obstacle_center),
    ca.DM([obstacle_radius]),
)

print("parameter vector length:", p_num.numel())
print("obstacle clearance lower bound:", 0.0)


## Solve From Multiple NLP Initial Guesses

Because this problem tracks only end-effector position, the UR5 has redundant degrees of freedom. Different feasible initial guesses can converge to different joint configurations while still satisfying the same task and nonlinear obstacle constraint.


In [ ]:
initial_guesses = [
    q0,
    np.clip(q0 + np.array([1.0, -1.0, 0.8, -0.6, 0.5, -0.4]), q_min + 1e-3, q_max - 1e-3),
    np.clip(q0 + np.array([-1.0, 0.8, -0.8, 0.7, -0.5, 0.4]), q_min + 1e-3, q_max - 1e-3),
    np.clip(q0 + np.array([0.7, 0.6, -1.0, 0.9, -0.6, 0.5]), q_min + 1e-3, q_max - 1e-3),
]

solution_records = []
for i, guess in enumerate(initial_guesses):
    sol = ik_nlp_solver(
        x0=guess,
        p=p_num,
        lbx=q_min,
        ubx=q_max,
        lbg=[0.0],
        ubg=[ca.inf],
    )
    q_sol = np.array(sol["x"]).reshape(-1)
    p_sol = np.array(ee_position_fun(q_sol)).reshape(-1)
    p_guess = np.array(ee_position_fun(guess)).reshape(-1)
    position_error = np.linalg.norm(p_sol - desired_position)
    obstacle_distance = np.linalg.norm(p_sol - obstacle_center)
    within_limits = bool(np.all(q_sol >= q_min - 1e-8) and np.all(q_sol <= q_max + 1e-8))

    solution_records.append({
        "start_id": i,
        "q0": guess,
        "q": q_sol,
        "p0": p_guess,
        "p": p_sol,
        "position_error": position_error,
        "obstacle_distance": obstacle_distance,
        "objective": float(sol["f"]),
        "within_limits": within_limits,
        "status": ik_nlp_solver.stats()["return_status"],
    })

for record in solution_records:
    print(f"start {record['start_id']}: status = {record['status']}")
    print(f"  objective        = {record['objective']:.3e}")
    print(f"  position error   = {record['position_error']:.3e}")
    print(f"  obstacle distance= {record['obstacle_distance']:.3e} >= {obstacle_radius:.3e}")
    print(f"  within limits    = {record['within_limits']}")
    print(f"  q solution       = {np.round(record['q'], 3)}")


## Plots

The plots show the NLP solution reached from each initial guess: task error, final joint configurations, and end-effector positions.


In [ ]:
fig = plt.figure(figsize=(15, 4))

start_ids = [record["start_id"] for record in solution_records]
position_errors = [record["position_error"] for record in solution_records]
objectives = [record["objective"] for record in solution_records]
q_solutions = np.vstack([record["q"] for record in solution_records])
p_initial = np.vstack([record["p0"] for record in solution_records])
p_solutions = np.vstack([record["p"] for record in solution_records])

ax1 = fig.add_subplot(1, 3, 1)
ax1.semilogy(start_ids, position_errors, marker="o", label="position error")
ax1.semilogy(start_ids, objectives, marker="s", label="objective")
ax1.set_xlabel("initial guess id")
ax1.set_ylabel("value")
ax1.grid(True)
ax1.legend()

ax2 = fig.add_subplot(1, 3, 2)
for i, q_sol in enumerate(q_solutions):
    ax2.plot(np.arange(model.nq), q_sol, marker="o", label=f"start {i}")
ax2.set_xlabel("joint")
ax2.set_ylabel("joint position")
ax2.grid(True)
ax2.legend(fontsize=8)

ax3 = fig.add_subplot(1, 3, 3, projection="3d")
ax3.scatter(*start_position, label="neutral start", s=60)
ax3.scatter(*desired_position, label="desired", s=70)
ax3.scatter(*obstacle_center, label="obstacle center", s=70)
for i in range(len(solution_records)):
    ax3.plot(
        [p_initial[i, 0], p_solutions[i, 0]],
        [p_initial[i, 1], p_solutions[i, 1]],
        [p_initial[i, 2], p_solutions[i, 2]],
        marker="o",
        label=f"start {i} to solution",
    )
ax3.set_xlabel("x")
ax3.set_ylabel("y")
ax3.set_zlabel("z")
ax3.legend(fontsize=7)

plt.tight_layout()
plt.show()


## Exercise

1. Move the spherical keep-out region and observe how the NLP solution changes.
2. Add a second nonlinear keep-out constraint of the same form.
3. Change `posture_weight` and observe whether the final configuration changes.
4. Extension: add orientation tracking by building a symbolic pose-error term.
